# Targeted fine-tuning — Leeds parking (run 2, rebalanced)

**What this asks.** The first fine-tuning run fed the model every Leeds label and let it work
things out. It raised IoU by 0.128 but bought precision with recall, and because the model
contracted everywhere, the fall in standalone false positives cannot be attributed to any
particular confusion class. This notebook closes that loop: it uses the Chapter 4 error
typology to build the supervision, then measures whether the removal is *selective*.

**What changed after run 1.** Run 1 reproduced the baseline exactly and showed the targeting
does reach its categories (selectivity gap +9.7 → +16.1) with boundary error left untouched by
design. But its recall fell further than the generic model's, and the log showed why — two
configuration faults rather than a property of the task:

* **The counterweight was ~4× too weak.** Hard-negative pixels carried 4.04% of the gradient
  mass and the false-negative counterweight only 0.93% — a 4.33 : 1 tilt toward predicting
  less. `W_FN` is now *computed* so that the false-negative mass matches the total upweighted
  false-positive mass, instead of being guessed.
* **Six epochs was the wrong budget and IoU picked the wrong checkpoint.** Validation recall
  was still climbing at epoch 6 (0.53 → 0.71 → 0.73 → 0.76 → 0.79), and epoch 5 held 0.063
  more recall than the selected epoch 3 for 0.0018 less IoU. Training now runs 12 epochs, keeps
  every candidate checkpoint, and evaluates **both** the best-IoU epoch and the best-recall
  epoch within an IoU tolerance — so the selection rule is measured rather than assumed.

**Everything heavy is cached to Drive and resumable.** Category rasters, weight codes, per-epoch
checkpoints, optimiser state and per-arm evaluation results all survive a disconnect. After a
drop, just Run all: each stage checks its own output first.

**Nothing in `fine-tuning/` is modified.** `modeling.py` and `patch_data.py` are imported from
it read-only so the preprocessing contract cannot drift.

## 1. Configuration, Drive and GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, shutil, subprocess, json, math

REPO_URL   = 'https://github.com/hou1020/Parking.git'
BRANCH     = 'main'
REPO       = Path('/content/Parking')
FT         = REPO / 'fine-tuning'            # read-only reuse
TG         = REPO / 'targeted-finetuning'

# The first experiment's Drive folder is READ ONLY here: we borrow its checkpoint and its
# prepared patches.  Everything this notebook produces goes to its own folder.
GENERIC_RUN = Path('/content/drive/MyDrive/Parking_finetuning_run')
RUN1        = Path('/content/drive/MyDrive/Parking_targeted_run')    # run 1, READ ONLY
RUN         = Path('/content/drive/MyDrive/Parking_targeted_run2')   # this run's outputs
CACHE       = RUN / 'cache'
EPOCH_DIR   = RUN / 'epochs'
EVAL_CACHE  = RUN / 'eval_cache'
for d in (RUN, CACHE, EPOCH_DIR, EVAL_CACHE):
    d.mkdir(parents=True, exist_ok=True)

# Run 1's folder is never written to: its log, checkpoint and tables are the record of the
# mistuned run and are still worth keeping.  Reusable caches are searched across both.
CACHE_ROOTS = [RUN, RUN1, GENERIC_RUN]

def find_cache(rel):
    for root in CACHE_ROOTS:
        p = root / rel
        if p.exists():
            return p
    return None

# --- shared with the generic run so the comparison stays controlled -------------------
TRAIN_BATCH, EVAL_BATCH, LR, SEED, NUM_WORKERS = 2, 4, 2e-5, 42, 2

# --- run 2 changes -------------------------------------------------------------------
EPOCHS   = 12       # run 1 stopped at 6 while validation recall was still climbing
IOU_TOL  = 0.02     # an epoch within this of the best IoU counts as a candidate operating point
RESUME   = True     # pick training up from last.ckpt after a disconnect

# --- typology-driven per-pixel loss weights ------------------------------------------
# Codes written by section 7:
#   0 ordinary pixel (includes boundary/dilation FP - deliberately NOT upweighted)
#   1 standalone FP that no layer explains
#   2 false negative (missed parking)
#   3 standalone FP on a precise layer   (road / curtilage / OSM parking / sports)
#   4 standalone FP on a broad land-use layer (brownfield / industrial)
W_ORDINARY, W_FP_OTHER, W_FP_PRECISE, W_FP_BROAD = 1.0, 1.0, 5.0, 2.0
# W_FN is NOT set here.  Section 8 computes it from the actual code composition so that the
# false-negative gradient mass equals the total upweighted false-positive mass.  Run 1 guessed
# 3.0 and ended up 4.33 : 1 against recall.  Set BALANCE_FN = False to use W_FN_MANUAL instead.
BALANCE_FN  = True
W_FN_MANUAL = 13.0

DILATION_M   = 5.0    # same working threshold as Chapter 4
EXTRA_ROAD_M = 6.0    # same road widening as fp_analysis.py
CURTILAGE_M  = 8.0    # buildings buffered outward: private forecourt / driveway proxy
PIXEL_M      = 0.25
CELL_PX      = 4000

FORCE_REBUILD_LAYERS = False
FORCE_REBUILD_CODES  = False
FORCE_RETRAIN        = False
FORCE_REEVALUATE     = False

# NOTE: torch / numpy are deliberately NOT imported here.  Section 2 installs packages, and
# anything already loaded into this process would keep its old C extensions while pip
# replaces the files on disk.  That mismatch is what produces
#     AttributeError: module 'numpy._core._multiarray_umath' has no attribute ...
print('run folder:', RUN)

## 2. Dependencies

`transformers` is pinned to the version the released checkpoint's key names belong to.

Two rules keep this cell from breaking the runtime:

* **no `--upgrade`** — Colab already ships working numpy, scipy, pandas, Pillow and torch, and
  upgrading them churns numpy for no benefit;
* **install before importing** — and if numpy or scipy did change anyway (a dependency of
  geopandas or rasterio can force it), the cell restarts the runtime itself. Colab will
  reconnect; just run all cells again from the top and this cell will be a no-op.

In [ ]:
import importlib.metadata as md

def ver(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

if shutil.which('git-lfs') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'git-lfs'], check=True)

todo = []
if ver('transformers') != '4.57.1':
    todo.append('transformers==4.57.1')
for pkg in ['geopandas', 'rasterio', 'tifffile', 'huggingface_hub', 'shapely']:
    if ver(pkg) is None:
        todo.append(pkg)

if todo:
    before = (ver('numpy'), ver('scipy'))
    print('installing:', ' '.join(todo))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *todo], check=True)
    after = (ver('numpy'), ver('scipy'))
    if before != after:
        print(f'numpy/scipy changed {before} -> {after}')
        print('RESTARTING THE RUNTIME. When it reconnects, Run all from the top.')
        os.kill(os.getpid(), 9)
else:
    print('all dependencies already satisfied')

assert ver('transformers') == '4.57.1', ver('transformers')
print('transformers', ver('transformers'), '| geopandas', ver('geopandas'),
      '| rasterio', ver('rasterio'), '| numpy', ver('numpy'), '| scipy', ver('scipy'))

# Safe to import heavy libraries only now that the environment is settled.
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))

## 3. Clone the repository

LFS smudging is off during clone; only the 100 source TIFFs are pulled. The reference layers
(`analysis/ref_cache.gpkg`, `analysis/osm_extra.gpkg`, the OS Greenspace shapefile and the 100
`osm_cache` road/building tiles) are ordinary tracked files and arrive with the clone.

This is the one heavy stage that cannot be cached to Drive — the TIFFs are far too large. If
the runtime was recycled it costs 10–20 minutes; if only the session restarted, the existing
clone is reused and this is a fast `git pull`.

In [ ]:
env = os.environ.copy(); env['GIT_LFS_SKIP_SMUDGE'] = '1'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)],
                   env=env, check=True)
else:
    print('existing clone found - updating in place')
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)

subprocess.run(['git', '-C', str(REPO), 'lfs', 'install', '--local'], check=True)
subprocess.run(['git', '-C', str(REPO), 'lfs', 'pull',
                '--include=parking-lot-mapping-tool/files/tif/**'], check=True)

TIF_ROOT = REPO / 'parking-lot-mapping-tool' / 'files' / 'tif'
n_tif = len(list(TIF_ROOT.rglob('*.tif')))
print('source TIFFs:', n_tif)
assert n_tif >= 100, 'LFS pull incomplete'

for p in ['analysis/ref_cache.gpkg', 'analysis/osm_extra.gpkg',
          'fine-tuning/leeds_manual.gpkg', 'fine-tuning/leeds_grid.gpkg', 'fine-tuning/split.csv']:
    assert (REPO / p).exists(), f'missing {p}'
print('reference layers present')

sys.path.insert(0, str(FT))          # modeling.py / patch_data.py, imported read-only
TG.mkdir(parents=True, exist_ok=True)

## 4. Prepared patches

Reuses the masks and `patch_index.csv` built by the first experiment. Restored from whichever
Drive cache exists — this run's own, or the generic run's. If neither is present they are
rebuilt from the committed GPKGs (deterministic, identical 2,256 / 3,200 split) and then
**cached into this run's folder**, so a later disconnect never rebuilds them again.

In [ ]:
def prepared_ok(root):
    root = Path(root)
    return ((root / 'patch_index.csv').exists()
            and len(list((root / 'patches/train/Masks').glob('*.png'))) == 2256
            and len(list((root / 'patches/test/Masks').glob('*.png'))) == 3200)

OWN_CACHE     = RUN / 'prepared_data'
GENERIC_CACHE = find_cache('prepared_data')

def restore(src):
    shutil.copy2(src / 'patch_index.csv', FT / 'patch_index.csv')
    shutil.copytree(src / 'patches', FT / 'patches', dirs_exist_ok=True)

if prepared_ok(FT):
    print('prepared data already in runtime')
elif prepared_ok(OWN_CACHE):
    restore(OWN_CACHE);     print("restored from this run's Drive cache")
elif GENERIC_CACHE and prepared_ok(GENERIC_CACHE):
    restore(GENERIC_CACHE); print('restored from', GENERIC_CACHE)
else:
    print('building patches from the committed GPKGs (a few minutes) ...')
    subprocess.run([sys.executable, 'make_patches.py'], cwd=str(FT), check=True)

assert prepared_ok(FT), 'patch preparation failed'

if not prepared_ok(OWN_CACHE):
    print('caching prepared data to Drive for future runs ...')
    OWN_CACHE.mkdir(parents=True, exist_ok=True)
    shutil.copy2(FT / 'patch_index.csv', OWN_CACHE / 'patch_index.csv')
    shutil.copytree(FT / 'patches', OWN_CACHE / 'patches', dirs_exist_ok=True)
    print('cached')

import pandas as pd, numpy as np
IDX   = pd.read_csv(FT / 'patch_index.csv')
SPLIT = pd.read_csv(FT / 'split.csv')
print(IDX.groupby('split').size().to_string())
print('cells:', SPLIT.groupby('split').size().to_dict())

## 5. Checkpoints

Arm A is the released model. Arm B is the first experiment's generic fine-tune — optional; if its checkpoint is not in Drive the notebook runs the remaining arms and says so.

In [ ]:
from huggingface_hub import hf_hub_download

ZERO_SHOT = find_cache('best_model.ckpt')
if ZERO_SHOT is None or ZERO_SHOT.stat().st_size < 1_000_000_000:
    hf_hub_download(repo_id='UTEL-UIUC/SegFormer-large-parking',
                    filename='best_model.ckpt', local_dir=str(RUN))
    ZERO_SHOT = RUN / 'best_model.ckpt'
assert ZERO_SHOT.stat().st_size > 1_000_000_000
print(f'arm A  zero-shot  : {ZERO_SHOT}  ({ZERO_SHOT.stat().st_size/1e9:.3f} GB)')

GENERIC_CKPT = GENERIC_RUN / 'finetuned.ckpt'
HAVE_GENERIC = GENERIC_CKPT.exists() and GENERIC_CKPT.stat().st_size > 1_000_000
print('arm B  generic FT :', GENERIC_CKPT if HAVE_GENERIC else 'NOT FOUND - arm B skipped')
print('arms C/D          : selected from this run in section 8')

## 6. Category rasters

The six layers of §4.2 are rasterised onto each cell's own 4,000 × 4,000 grid and packed as
bitplanes in one `uint8` array per cell. `curtilage` is new here: buildings buffered outward by
8 m and the footprints removed, standing in for the private forecourts and driveways that the
sampling estimated at 20.2% of the unexplained residual. It is a proxy, and §4.2's caveat
applies to it more than to any other layer — attribution is by location, not by inspection.

**Cached to Drive**, and now written **one cell at a time** straight into the compressed
archive, so peak memory stays at ~16 MB instead of holding all 100 cells (1.6 GB) at once.

In [ ]:
import geopandas as gpd
import zipfile, io
from shapely.ops import unary_union
from shapely.geometry import box
from rasterio.features import rasterize
from rasterio.transform import from_origin
import warnings; warnings.filterwarnings('ignore')

LAYERS = ['building', 'osm_parking', 'sports', 'road_wide', 'curtilage', 'brownfield', 'industrial']
BIT    = {n: i for i, n in enumerate(LAYERS)}
PRECISE = ['osm_parking', 'sports', 'road_wide', 'curtilage']     # -> code 3
BROAD   = ['brownfield', 'industrial']                            # -> code 4
# Peeling order for attribution: most specific evidence first, exactly as fp_analysis.py,
# with curtilage inserted next to road_adjacent because both are proximity rules.
PEEL = ['building', 'osm_parking', 'sports', 'road_wide', 'curtilage', 'brownfield', 'industrial']

LAYER_NPZ = CACHE / 'category_layers.npz'


def _dissolve(gdf):
    g = gdf.to_crs(27700).copy()
    g = g[g.geometry.type.isin(['Polygon', 'MultiPolygon'])]
    if not len(g):
        return None
    g['geometry'] = g.geometry.buffer(0)
    return unary_union(g.geometry.values)


def build_layer_geometries():
    print('loading reference layers ...')
    ref = gpd.read_file(REPO / 'analysis/ref_cache.gpkg').to_crs(27700)
    buildings = _dissolve(ref[ref['grp'] == 'buildings'])
    roads     = _dissolve(ref[ref['grp'] == 'roads'])

    ex  = gpd.read_file(REPO / 'analysis/osm_extra.gpkg').to_crs(27700)
    grp = lambda k: _dissolve(ex[ex['grp'] == k]) if (ex['grp'] == k).any() else None

    gs = gpd.read_file(REPO / 'analysis/OS Open Greenspace (ESRI Shape File) SE/data/SE_GreenspaceSite.shp').to_crs(27700)
    gs_sports = _dissolve(gs[gs['function'].isin(['Tennis Court', 'Other Sports Facility', 'Play Space'])])

    pitch  = grp('pitch')
    sports = unary_union([g for g in (gs_sports, pitch) if g is not None])

    geoms = {
        'building':    buildings,
        'osm_parking': grp('osm_parking'),
        'sports':      sports,
        'road_wide':   roads.buffer(EXTRA_ROAD_M) if roads is not None else None,
        'curtilage':   buildings.buffer(CURTILAGE_M).difference(buildings) if buildings is not None else None,
        'brownfield':  grp('brownfield_bare'),
        'industrial':  grp('industrial_yard'),
    }
    for k, v in geoms.items():
        print(f'  {k:<12} {"ok" if v is not None and not v.is_empty else "EMPTY"}')
    return geoms


def build_category_rasters(path):
    """Stream one cell at a time into the .npz so peak memory is a single 16 MB array."""
    geoms = build_layer_geometries()
    parts = {k: gpd.GeoSeries([g], crs=27700).explode(index_parts=False).reset_index(drop=True)
             for k, g in geoms.items() if g is not None and not g.is_empty}
    sidx = {k: v.sindex for k, v in parts.items()}

    tmp = Path(str(path) + '.partial')
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for n, row in enumerate(SPLIT.itertuples(), 1):
            tf = from_origin(row.left, row.top, PIXEL_M, PIXEL_M)
            bb = box(row.left, row.bottom, row.right, row.top)
            packed = np.zeros((CELL_PX, CELL_PX), dtype=np.uint8)
            for name, gs_parts in parts.items():
                hit = sidx[name].query(bb, predicate='intersects')
                if len(hit) == 0:
                    continue
                shapes = [(g, 1) for g in gs_parts.iloc[hit].values if g is not None and not g.is_empty]
                if not shapes:
                    continue
                r = rasterize(shapes, out_shape=(CELL_PX, CELL_PX), transform=tf, fill=0,
                              dtype='uint8', all_touched=False)
                packed |= (r.astype(np.uint8) << BIT[name])
            buf = io.BytesIO()
            np.lib.format.write_array(buf, packed, allow_pickle=False)
            zf.writestr(f'{row.cell}.npy', buf.getvalue())
            del packed, buf
            if n % 20 == 0 or n == len(SPLIT):
                print(f'  rasterised {n}/{len(SPLIT)} cells')
    tmp.replace(path)     # atomic: a half-written archive is never left as the cache


_cached = find_cache('cache/category_layers.npz')
if _cached and not FORCE_REBUILD_LAYERS:
    LAYER_NPZ = _cached
    print('using cached category rasters:', LAYER_NPZ)
else:
    build_category_rasters(LAYER_NPZ)

# Kept lazy on purpose: holding all 100 cells would pin ~1.6 GB of RAM, and each cell is
# touched only twice in the whole notebook.  NpzFile decompresses on access.
CATEGORY = np.load(LAYER_NPZ)
assert len(CATEGORY.files) == 100, len(CATEGORY.files)
_c = CATEGORY[CATEGORY.files[0]]
print('cells:', len(CATEGORY.files), '| shape', _c.shape, '| coverage % in cell 0:',
      {n: round(100 * float(((_c >> BIT[n]) & 1).mean()), 1) for n in LAYERS})
del _c

## 7. Zero-shot error maps → per-pixel weight codes

Run the released model over the **training half only** and turn its own mistakes into
supervision. Two choices carry the design:

* **Boundary FP is left at weight 1.** Upweighting FP that lies within 5 m of a real car park
  is precisely how a model is taught to draw everything smaller. Run 1 confirmed this works:
  boundary FP and FN erosion were both unchanged between the generic and targeted models.
  Only *standalone* FP becomes a hard negative.
* **FN is upweighted**, by an amount section 8 computes rather than guesses.

The held-out 50 cells are never read here. The finished codes are **zipped to Drive**, so a
disconnect costs a 30-second unzip instead of a 15-minute rebuild.

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from PIL import Image
from scipy import ndimage
from modeling import make_processor, make_model, load_checkpoint
from patch_data import SourcePatchDataset, TileBatchSampler

CODE_ROOT = FT / 'patches' / 'train' / 'Codes'     # fast local reads during training
CODES_ZIP = RUN / 'weight_codes.zip'               # durable copy on Drive
device = torch.device('cuda')
DIL_PX = DILATION_M / PIXEL_M


def assemble(pairs):
    """Rebuild a full 4000x4000 boolean cell from its 64 (row, array) pairs."""
    cell = np.zeros((CELL_PX, CELL_PX), dtype=bool)
    for r, arr in pairs:
        vh, vw = r['valid_h'], r['valid_w']
        cell[r['row_off']:r['row_off'] + vh, r['col_off']:r['col_off'] + vw] = arr[:vh, :vw]
    return cell


@torch.no_grad()
def predict_rows(model, rows, patch_root, batch_size=EVAL_BATCH):
    ds = SourcePatchDataset(rows, patch_root, str(TIF_ROOT), make_processor(), return_index=True)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS,
                    pin_memory=True)
    preds = [None] * len(rows)
    for batch, ids in dl:
        logits = model(pixel_values=batch['pixel_values'].to(device, non_blocking=True))[0]
        up = F.interpolate(logits, size=batch['labels'].shape[-2:], mode='bilinear',
                           align_corners=False)
        p = up.argmax(1).cpu().numpy().astype(bool)
        for b, i in enumerate(ids.tolist()):
            preds[i] = p[b]
    return preds


def build_weight_codes():
    CODE_ROOT.mkdir(parents=True, exist_ok=True)
    model = load_checkpoint(make_model(), str(ZERO_SHOT)).to(device).eval()
    train_rows = IDX[(IDX['split'] == 'train') & IDX['kept']].sort_values(
        ['cell', 'row_off', 'col_off']).to_dict('records')

    stats = {c: 0 for c in range(5)}
    cells = sorted({r['cell'] for r in train_rows})
    for n, cell in enumerate(cells, 1):
        rows = [r for r in train_rows if r['cell'] == cell]
        preds = predict_rows(model, rows, str(FT / 'patches' / 'train'))

        masks = []
        for r in rows:
            with Image.open(FT / 'patches' / 'train' / 'Masks' / r['name']) as im:
                masks.append(np.array(im) == 1)
        ref  = assemble(list(zip(rows, masks)))
        pred = assemble(list(zip(rows, preds)))

        fp, fn = pred & ~ref, ~pred & ref
        if ref.any():
            standalone = fp & (ndimage.distance_transform_edt(~ref) > DIL_PX)
        else:
            standalone = fp

        bits = CATEGORY[cell]
        precise = np.zeros_like(ref); broad = np.zeros_like(ref)
        for name in PRECISE:
            precise |= ((bits >> BIT[name]) & 1).astype(bool)
        for name in BROAD:
            broad |= ((bits >> BIT[name]) & 1).astype(bool)

        codeplane = np.zeros((CELL_PX, CELL_PX), dtype=np.uint8)
        codeplane[standalone] = 1
        codeplane[standalone & broad] = 4
        codeplane[standalone & precise] = 3      # precise beats broad where they overlap
        codeplane[fn] = 2                        # FP and FN are disjoint

        for r in rows:
            vh, vw = r['valid_h'], r['valid_w']
            tile = np.zeros((512, 512), dtype=np.uint8)
            tile[:vh, :vw] = codeplane[r['row_off']:r['row_off'] + vh,
                                       r['col_off']:r['col_off'] + vw]
            Image.fromarray(tile).save(CODE_ROOT / r['name'])
        for c in range(5):
            stats[c] += int((codeplane == c).sum())
        if n % 10 == 0 or n == len(cells):
            print(f'  {n}/{len(cells)} training cells')

    del model; torch.cuda.empty_cache()
    tot = sum(stats.values())
    names = {0: 'ordinary (incl. boundary FP)', 1: 'standalone FP, unattributed',
             2: 'false negative', 3: 'standalone FP, precise layer',
             4: 'standalone FP, broad layer'}
    print('\nweight-code composition over the training half')
    for c in range(5):
        print(f'  {c}  {names[c]:<32} {stats[c]*PIXEL_M**2/1e6:8.4f} km²  {100*stats[c]/tot:6.3f}%')
    pd.DataFrame([{'code': c, 'meaning': names[c], 'km2': stats[c]*PIXEL_M**2/1e6,
                   'pct': 100*stats[c]/tot} for c in range(5)]).to_csv(RUN / 'weight_codes.csv',
                                                                      index=False)

    print('zipping codes to Drive ...')
    tmp = Path(str(CODES_ZIP) + '.partial')
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(CODE_ROOT.glob('*.png')):
            zf.write(p, p.name)
    tmp.replace(CODES_ZIP)
    print(f'cached: {CODES_ZIP} ({CODES_ZIP.stat().st_size/1e6:.1f} MB)')


def codes_present():
    return CODE_ROOT.exists() and len(list(CODE_ROOT.glob('*.png'))) == 2256

_zip = find_cache('weight_codes.zip')
if codes_present() and not FORCE_REBUILD_CODES:
    print('weight codes already in runtime')
elif _zip and not FORCE_REBUILD_CODES:
    print('restoring weight codes from', _zip)
    CODE_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(_zip) as zf:
        zf.extractall(CODE_ROOT)
    print('restored')
else:
    build_weight_codes()

assert codes_present(), 'weight codes incomplete'
if not (RUN / 'weight_codes.csv').exists():
    _src = find_cache('weight_codes.csv')
    if _src:
        shutil.copy2(_src, RUN / 'weight_codes.csv')
if (RUN / 'weight_codes.csv').exists():
    display(pd.read_csv(RUN / 'weight_codes.csv'))

## 8. Targeted fine-tuning

Weighted cross-entropy, reproducing Hugging Face's own loss path (logits bilinearly upsampled
to label size, `ignore_index=255`) but with `reduction='none'` so the weight map can be applied.
The loss is normalised by the **sum of weights**, not the pixel count — dividing by count would
make the weighted loss numerically larger and effectively raise the learning rate, confounding
the comparison with the thing being tested.

**`W_FN` is computed, not guessed.** Run 1 set it to 3.0, which gave the hard negatives 4.33×
the gradient mass of the counterweight and guaranteed a contracting model. It is now solved so
that false-negative mass equals total upweighted false-positive mass, using the composition
measured in section 7.

**Resumable.** After every epoch the notebook writes the log, a candidate checkpoint, and a
`last.ckpt` holding model, optimiser, gradient scaler, history and the batch sampler's epoch
counter — so a resumed run reproduces the same data ordering it would have had. Candidate
checkpoints outside `IOU_TOL` of the running best are deleted as they fall out of contention,
which keeps Drive usage near 2 GB rather than 4.

In [ ]:
import random, time
from modeling import BASE_MODEL, PATCH_SIZE

LOG_CSV = RUN / 'targeted_log.csv'
LAST    = RUN / 'last.ckpt'
BEST    = RUN / 'targeted.ckpt'

# The balance must be struck over the pixels the loss actually sees.  Sampling kept every
# positive patch but only an equal number of empty ones, so parking - and therefore FN - is
# far denser in the fit patches than across whole cells.  Run 1 balanced against the whole-cell
# composition, which is one reason the counterweight came out too light.
_tr = IDX[(IDX['split'] == 'train') & IDX['kept']]
_counts = np.zeros(5, dtype=np.int64)
for _n, _name in enumerate(_tr['name'], 1):
    with Image.open(CODE_ROOT / _name) as _im:
        _counts += np.bincount(np.asarray(_im).ravel(), minlength=5)[:5]
    if _n % 500 == 0:
        print(f'  scanned {_n}/{len(_tr)} code tiles')
comp = pd.Series(100.0 * _counts / _counts.sum(), index=range(5))
print('\ncode composition over the fit patches (what the loss sees)')
for c in range(5):
    print(f'  {c}  {comp[c]:7.3f}%')

if BALANCE_FN:
    fp_mass = comp[1]*W_FP_OTHER + comp[3]*W_FP_PRECISE + comp[4]*W_FP_BROAD
    W_FN = float(fp_mass / comp[2]) if comp[2] > 0 else W_FN_MANUAL
    print(f'\nupweighted FP mass {fp_mass:.3f} | FN pixels {comp[2]:.3f}%  ->  W_FN = {W_FN:.2f}')
else:
    W_FN = W_FN_MANUAL
    print(f'\nW_FN fixed at {W_FN}')

WEIGHTS = torch.tensor([W_ORDINARY, W_FP_OTHER, W_FN, W_FP_PRECISE, W_FP_BROAD],
                       dtype=torch.float32, device=device)
_mass = [comp[c] * float(WEIGHTS[c]) for c in range(5)]
print('\ngradient mass by code')
for c in range(5):
    print(f'  {c}  pixels {comp[c]:7.3f}%   weight {float(WEIGHTS[c]):5.1f}   '
          f'mass {100*_mass[c]/sum(_mass):6.2f}%')
print(f'suppression (1,3,4) : recovery (2)  =  '
      f'{sum(_mass[c] for c in (1,3,4)):.3f} : {_mass[2]:.3f}')


class CodedPatchDataset(torch.utils.data.Dataset):
    """SourcePatchDataset plus the per-pixel weight code for the same patch."""
    def __init__(self, rows, patch_root, code_root, processor):
        self.base = SourcePatchDataset(rows, patch_root, str(TIF_ROOT), processor,
                                       ignore_padding=True)
        self.rows, self.code_root = rows, Path(code_root)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        enc = self.base[i]
        with Image.open(self.code_root / self.rows[i]['name']) as im:
            enc['weight_code'] = torch.from_numpy(np.array(im).astype(np.int64))
        return enc


def weighted_loss(logits, labels, codes):
    up = F.interpolate(logits, size=labels.shape[-2:], mode='bilinear', align_corners=False)
    valid = labels != 255
    safe = labels.clone(); safe[~valid] = 0
    ce = F.cross_entropy(up.float(), safe.long(), reduction='none')
    w = WEIGHTS[codes] * valid
    return (ce * w).sum() / w.sum().clamp(min=1.0)


@torch.no_grad()
def val_scores(model, loader):
    model.eval(); tp = fp = fn = 0
    for batch in loader:
        gt = batch['labels'].to(device)
        logits = model(pixel_values=batch['pixel_values'].to(device))[0]
        up = F.interpolate(logits, size=gt.shape[-2:], mode='bilinear', align_corners=False)
        pr = up.argmax(1); ok = gt != 255
        tp += int(((pr == 1) & (gt == 1) & ok).sum())
        fp += int(((pr == 1) & (gt == 0) & ok).sum())
        fn += int(((pr != 1) & (gt == 1) & ok).sum())
    d = lambda a, b: a / b if b else 0.0
    return d(tp, tp + fp), d(tp, tp + fn), d(tp, tp + fp + fn)


def prune_epoch_ckpts(hist):
    """Keep only epochs still within IOU_TOL of the best - candidate operating points."""
    done = [h for h in hist if h['epoch'] > 0 and h['val_parking_iou'] is not None]
    if not done:
        return
    top = max(h['val_parking_iou'] for h in done)
    keep = {h['epoch'] for h in done if h['val_parking_iou'] >= top - IOU_TOL}
    for f in EPOCH_DIR.glob('epoch_*.ckpt'):
        if int(f.stem.split('_')[1]) not in keep:
            f.unlink()


def train_targeted():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    rng = np.random.default_rng(SEED)

    tr = IDX[(IDX['split'] == 'train') & IDX['kept']]
    cells = np.array(sorted(tr['cell'].unique())); rng.shuffle(cells)
    n_val = max(1, int(round(0.2 * len(cells))))
    val_cells, fit_cells = set(cells[:n_val]), set(cells[n_val:])
    fit_rows = tr[tr['cell'].isin(fit_cells)].to_dict('records')
    val_rows = tr[tr['cell'].isin(val_cells)].to_dict('records')
    print(f'\nfit {len(fit_cells)} cells / {len(fit_rows)} patches | '
          f'val {len(val_cells)} cells / {len(val_rows)} patches')
    pd.DataFrame([{'cell': c, 'role': 'fit' if c in fit_cells else 'validation'}
                  for c in sorted(fit_cells | val_cells)]).to_csv(RUN / 'fit_val_split.csv', index=False)

    proc = make_processor()
    root = str(FT / 'patches' / 'train')
    fit_ds = CodedPatchDataset(fit_rows, root, CODE_ROOT, proc)
    val_ds = SourcePatchDataset(val_rows, root, str(TIF_ROOT), proc, ignore_padding=True)
    fit_sampler = TileBatchSampler(fit_rows, TRAIN_BATCH, SEED, True)
    fit_dl = DataLoader(fit_ds, batch_sampler=fit_sampler, num_workers=NUM_WORKERS, pin_memory=True)
    val_dl = DataLoader(val_ds, batch_size=EVAL_BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

    model = load_checkpoint(make_model(), str(ZERO_SHOT)).to(device)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR, eps=1e-8)
    scaler = torch.cuda.amp.GradScaler(enabled=True)

    start_epoch, best, best_epoch, hist = 1, -1.0, None, []
    if RESUME and LAST.exists() and not FORCE_RETRAIN:
        try:
            blob = torch.load(LAST, map_location='cpu', weights_only=False)
        except TypeError:                      # older torch has no weights_only kwarg
            blob = torch.load(LAST, map_location='cpu')
        model.load_state_dict(blob['state_dict']); model.to(device)
        opt.load_state_dict(blob['optimizer'])
        scaler.load_state_dict(blob['scaler'])
        hist, best, best_epoch = blob['history'], blob['best'], blob['best_epoch']
        start_epoch = blob['epoch'] + 1
        fit_sampler.epoch = blob['sampler_epoch']   # keep the data ordering it would have had
        print(f'resuming from epoch {start_epoch} (best so far {best:.4f} @ epoch {best_epoch})')
    else:
        p0, r0, i0 = val_scores(model, val_dl)
        print(f'zero-shot on validation cells: P {p0:.4f}  R {r0:.4f}  IoU {i0:.4f}')
        hist = [{'epoch': 0, 'train_loss': None, 'val_precision': round(p0, 4),
                 'val_recall': round(r0, 4), 'val_parking_iou': round(i0, 4), 'note': 'zero-shot'}]
        pd.DataFrame(hist).to_csv(LOG_CSV, index=False)

    for ep in range(start_epoch, EPOCHS + 1):
        model.train(); losses = []; t0 = time.time()
        for step, batch in enumerate(fit_dl, 1):
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=True):
                logits = model(pixel_values=batch['pixel_values'].to(device, non_blocking=True))[0]
            loss = weighted_loss(logits, batch['labels'].to(device, non_blocking=True),
                                 batch['weight_code'].to(device, non_blocking=True))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            losses.append(float(loss.detach()))
            if step % 200 == 0:
                print(f'  ep {ep} step {step}/{len(fit_dl)} loss {np.mean(losses[-200:]):.4f}')
        p, r, i = val_scores(model, val_dl)
        print(f'epoch {ep:2d}: loss {np.mean(losses):.4f} | val P {p:.4f} R {r:.4f} IoU {i:.4f}'
              f' | {time.time()-t0:.0f}s' + ('  <- best' if i > best else ''))
        hist.append({'epoch': ep, 'train_loss': round(float(np.mean(losses)), 4),
                     'val_precision': round(p, 4), 'val_recall': round(r, 4),
                     'val_parking_iou': round(i, 4), 'note': ''})

        # --- everything durable, every epoch -----------------------------------------
        pd.DataFrame(hist).to_csv(LOG_CSV, index=False)
        torch.save({'state_dict': model.state_dict(), 'epoch': ep, 'val_parking_iou': i,
                    'val_precision': p, 'val_recall': r, 'base_model': BASE_MODEL,
                    'processor_size': PATCH_SIZE, 'seed': SEED, 'weights': WEIGHTS.tolist()},
                   EPOCH_DIR / f'epoch_{ep:02d}.ckpt')
        if i > best:
            best, best_epoch = i, ep
            shutil.copy2(EPOCH_DIR / f'epoch_{ep:02d}.ckpt', BEST)
        prune_epoch_ckpts(hist)
        torch.save({'state_dict': model.state_dict(), 'optimizer': opt.state_dict(),
                    'scaler': scaler.state_dict(), 'epoch': ep, 'history': hist,
                    'best': best, 'best_epoch': best_epoch,
                    'sampler_epoch': fit_sampler.epoch}, LAST)

    print(f'\nbest epoch {best_epoch}, validation parking IoU {best:.4f}')
    del model; torch.cuda.empty_cache()


def training_complete():
    if not LOG_CSV.exists() or not BEST.exists():
        return False
    return int(pd.read_csv(LOG_CSV).epoch.max()) >= EPOCHS

if training_complete() and not FORCE_RETRAIN:
    print('training already complete for this epoch budget')
else:
    train_targeted()

LOG = pd.read_csv(LOG_CSV)
display(LOG)

## 9. Choosing the arms

Run 1's selected checkpoint was the best-IoU epoch, and the log then showed a later epoch
holding 0.063 more validation recall for 0.0018 less IoU. Rather than swap one arbitrary rule
for another, both are evaluated:

* **arm C** — highest validation IoU. The same rule the generic run used, so C vs B isolates
  the loss weighting.
* **arm D** — highest validation *recall* among epochs within `IOU_TOL` of the best IoU. This
  measures what the selection rule costs, holding the trained model fixed.

If C and D are the same epoch, arm D is skipped.

In [ ]:
trained = LOG[LOG.epoch > 0].dropna(subset=['val_parking_iou'])
best_iou_row = trained.loc[trained.val_parking_iou.idxmax()]
band = trained[trained.val_parking_iou >= best_iou_row.val_parking_iou - IOU_TOL]
best_rec_row = band.loc[band.val_recall.idxmax()]

print(f'best IoU    : epoch {int(best_iou_row.epoch):2d}  '
      f'IoU {best_iou_row.val_parking_iou:.4f}  R {best_iou_row.val_recall:.4f}')
print(f'best recall : epoch {int(best_rec_row.epoch):2d}  '
      f'IoU {best_rec_row.val_parking_iou:.4f}  R {best_rec_row.val_recall:.4f}'
      f'   (within {IOU_TOL} IoU of the best)')
print(f'\ncandidates kept on Drive: '
      f'{sorted(int(f.stem.split("_")[1]) for f in EPOCH_DIR.glob("epoch_*.ckpt"))}')

ARMS = [('A zero-shot', ZERO_SHOT)]
if HAVE_GENERIC:
    ARMS.append(('B generic FT', GENERIC_CKPT))
ARMS.append((f'C targeted e{int(best_iou_row.epoch)} (best IoU)', BEST))
if int(best_rec_row.epoch) != int(best_iou_row.epoch):
    alt = EPOCH_DIR / f'epoch_{int(best_rec_row.epoch):02d}.ckpt'
    if alt.exists():
        ARMS.append((f'D targeted e{int(best_rec_row.epoch)} (best recall)', alt))
    else:
        print(f'note: {alt.name} was pruned, arm D unavailable')

for n, p in ARMS:
    print(' ', n)

## 10. Category-resolved evaluation

Same 50 held-out cells, same preprocessing, for every arm. Each complete 1 km cell is
reassembled before any distance is measured, so patch seams are never mistaken for car-park
boundaries. FP is split into dilation (≤ 5 m of a label) and standalone, and the standalone
part is attributed by the peeling order of §4.2. FN is split against the *prediction*, not
against the reference edge.

**Each arm's raw counts are cached to Drive as soon as it finishes**, so a disconnect during
the third arm costs only that arm.

**Sanity check built in:** arm A must reproduce micro P 0.5190 / R 0.8819 / IoU 0.4853. If it
does not, something upstream has drifted and the comparison is void.

In [ ]:
TEST = IDX[(IDX['split'] == 'test') & IDX['kept']].sort_values(['cell', 'row_off', 'col_off'])
counts = TEST.groupby('cell').size()
assert len(counts) == 50 and (counts == 64).all(), 'held-out set incomplete'
TEST_ROWS = TEST.to_dict('records')
print(f'held-out: {len(TEST_ROWS)} patches / {len(counts)} cells')

BANDS_M = (2.0, 5.0, 10.0)


def score_arm(ckpt, name):
    model = load_checkpoint(make_model(), str(ckpt)).to(device).eval()
    per_cell, cat = {}, {k: 0 for k in PEEL + ['other']}
    bands = {m: dict(fp_dil=0, fp_std=0, fn_ero=0, fn_std=0) for m in BANDS_M}

    cells = sorted({r['cell'] for r in TEST_ROWS})
    for n, cell in enumerate(cells, 1):
        rows = [r for r in TEST_ROWS if r['cell'] == cell]
        preds = predict_rows(model, rows, str(FT / 'patches' / 'test'))
        masks = []
        for r in rows:
            with Image.open(FT / 'patches' / 'test' / 'Masks' / r['name']) as im:
                masks.append(np.array(im) == 1)
        ref  = assemble(list(zip(rows, masks)))
        pred = assemble(list(zip(rows, preds)))

        tp_m, fp_m, fn_m = pred & ref, pred & ~ref, ~pred & ref
        per_cell[cell] = [int(tp_m.sum()), int(fp_m.sum()), int(fn_m.sum())]

        d_ref  = ndimage.distance_transform_edt(~ref)  if ref.any()  else None
        d_pred = ndimage.distance_transform_edt(~pred) if pred.any() else None
        for m in BANDS_M:
            px = m / PIXEL_M
            if d_ref is not None:
                near = d_ref <= px
                bands[m]['fp_dil'] += int((fp_m & near).sum())
                bands[m]['fp_std'] += int((fp_m & ~near).sum())
            else:
                bands[m]['fp_std'] += int(fp_m.sum())
            if d_pred is not None:
                near = d_pred <= px
                bands[m]['fn_ero'] += int((fn_m & near).sum())
                bands[m]['fn_std'] += int((fn_m & ~near).sum())
            else:
                bands[m]['fn_std'] += int(fn_m.sum())

        rem = fp_m & (d_ref > DIL_PX) if d_ref is not None else fp_m.copy()
        bits = CATEGORY[cell]
        for layer in PEEL:
            hit = rem & ((bits >> BIT[layer]) & 1).astype(bool)
            cat[layer] += int(hit.sum()); rem &= ~hit
        cat['other'] += int(rem.sum())

        del d_ref, d_pred
        if n % 10 == 0 or n == len(cells):
            print(f'  [{name}] {n}/{len(cells)} cells')

    del model; torch.cuda.empty_cache()
    return per_cell, cat, bands


def arm_cache(name):
    return EVAL_CACHE / (''.join(ch if ch.isalnum() else '_' for ch in name) + '.json')


def score_arm_cached(ckpt, name):
    f = arm_cache(name)
    if f.exists() and not FORCE_REEVALUATE:
        blob = json.loads(f.read_text())
        print(f'{name}: restored from cache')
        return blob['per_cell'], blob['cat'], {float(k): v for k, v in blob['bands'].items()}
    print(f'\n=== scoring {name} ===')
    per_cell, cat, bands = score_arm(ckpt, name)
    f.write_text(json.dumps({'per_cell': per_cell, 'cat': cat,
                             'bands': {str(k): v for k, v in bands.items()}}))
    print(f'{name}: cached to {f.name}')
    return per_cell, cat, bands


def summarise(name, per_cell, cat, bands):
    a = PIXEL_M ** 2 / 1e6
    tp = sum(v[0] for v in per_cell.values())
    fp = sum(v[1] for v in per_cell.values())
    fn = sum(v[2] for v in per_cell.values())
    f = lambda x, y: x / y if y else float('nan')
    rows = [{'model': name, 'aggregation': 'micro',
             'precision': round(f(tp, tp+fp), 4), 'recall': round(f(tp, tp+fn), 4),
             'iou': round(f(tp, tp+fp+fn), 4), 'tp_km2': round(tp*a, 4),
             'fp_km2': round(fp*a, 4), 'fn_km2': round(fn*a, 4),
             'predicted_km2': round((tp+fp)*a, 4), 'reference_km2': round((tp+fn)*a, 4)}]
    pc = [(f(v[0], v[0]+v[1]), f(v[0], v[0]+v[2]), f(v[0], sum(v))) for v in per_cell.values() if sum(v)]
    rows.append({'model': name, 'aggregation': 'macro',
                 'precision': round(float(np.nanmean([x[0] for x in pc])), 4),
                 'recall':    round(float(np.nanmean([x[1] for x in pc])), 4),
                 'iou':       round(float(np.nanmean([x[2] for x in pc])), 4)})
    band_rows = [{'model': name, 'distance_m': m,
                  'fp_dilation_km2': round(b['fp_dil']*a, 4),
                  'fp_standalone_km2': round(b['fp_std']*a, 4),
                  'fn_erosion_km2': round(b['fn_ero']*a, 4),
                  'fn_standalone_km2': round(b['fn_std']*a, 4)} for m, b in bands.items()]
    cat_rows = [{'model': name, 'category': k, 'standalone_fp_km2': round(v*a, 4)}
                for k, v in cat.items()]
    return rows, band_rows, cat_rows


all_rows, all_bands, all_cats = [], [], []
for name, ckpt in ARMS:
    pc, cat, bands = score_arm_cached(ckpt, name)
    r, b, c = summarise(name, pc, cat, bands)
    all_rows += r; all_bands += b; all_cats += c

EVAL  = pd.DataFrame(all_rows)
BANDS = pd.DataFrame(all_bands)
CATS  = pd.DataFrame(all_cats)
EVAL.to_csv(RUN / 'evaluation_arms.csv', index=False)
BANDS.to_csv(RUN / 'boundary_bands_arms.csv', index=False)
CATS.to_csv(RUN / 'standalone_fp_by_category.csv', index=False)

z = EVAL[(EVAL.model == 'A zero-shot') & (EVAL.aggregation == 'micro')].iloc[0]
print(f'\nsanity check - arm A micro: P {z.precision} R {z.recall} IoU {z.iou}')
print('expected: P 0.5190  R 0.8819  IoU 0.4853')
if not (abs(z.precision-0.5190) < 0.002 and abs(z.recall-0.8819) < 0.002):
    print('*** MISMATCH - the pipeline has drifted; do not report the comparison ***')
else:
    print('reproduced - all runs share a baseline')

## 11. The tables

**Table 1** overall accuracy per arm — does the rebalanced weighting keep the recall run 1 lost?

**Table 2** standalone FP by category and the **removal rate** relative to zero-shot. Removal
rates roughly flat across categories mean the model simply contracted; a higher rate in the
targeted categories than elsewhere means the supervision reached what the typology named.

**Table 3** boundary bands — the check that FN loss is edge retraction or whole-lot failure.

In [ ]:
pd.set_option('display.width', 220)

print('=== Table 1  overall accuracy on the 50 held-out cells ===')
t1 = EVAL[EVAL.aggregation == 'micro'][
    ['model', 'precision', 'recall', 'iou', 'tp_km2', 'fp_km2', 'fn_km2',
     'predicted_km2', 'reference_km2']].reset_index(drop=True)
t1['pred_vs_ref_%'] = (100 * (t1.predicted_km2 - t1.reference_km2) / t1.reference_km2).round(1)
display(t1)
display(EVAL[EVAL.aggregation == 'macro'][['model', 'precision', 'recall', 'iou']]
        .reset_index(drop=True))

print('\n=== Table 2  standalone FP (>5 m) by category, and removal vs zero-shot ===')
piv = CATS.pivot(index='category', columns='model', values='standalone_fp_km2')
base = piv['A zero-shot']
t2 = pd.DataFrame({'zero_shot_km2': base})
for name, _ in ARMS[1:]:
    t2[f'{name}_km2'] = piv[name]
    t2[f'{name}_rm_%'] = (100 * (base - piv[name]) / base.replace(0, np.nan)).round(1)
order = [c for c in PEEL + ['other'] if c in t2.index]
t2 = t2.loc[order].round(4)
t2.loc['TOTAL'] = t2.loc[order].sum(numeric_only=True).round(4)
for name, _ in ARMS[1:]:
    t2.loc['TOTAL', f'{name}_rm_%'] = round(100 * (base.sum() - piv[name].sum()) / base.sum(), 1)
display(t2)

print('\n=== Table 3  boundary bands at 5 m ===')
display(BANDS[BANDS.distance_m == 5.0].reset_index(drop=True))

print('\n=== Selectivity read-out ===')
TARGETED_CATS = ['road_wide', 'curtilage', 'osm_parking', 'sports']
sel = []
for name, _ in ARMS[1:]:
    col = f'{name}_rm_%'
    tgt = t2.loc[[c for c in TARGETED_CATS if c in order], col].mean()
    oth = t2.loc[[c for c in order if c not in TARGETED_CATS], col].mean()
    sel.append({'arm': name, 'targeted_%': round(tgt, 1), 'other_%': round(oth, 1),
                'gap_pts': round(tgt - oth, 1)})
    print(f'{name}: targeted {tgt:.1f}%  others {oth:.1f}%  ->  gap {tgt-oth:+.1f} pts')
pd.DataFrame(sel).to_csv(RUN / 'selectivity.csv', index=False)

print('\n=== Where the FN sits (5 m) ===')
fnb = BANDS[BANDS.distance_m == 5.0].set_index('model')
for name, _ in ARMS:
    row = fnb.loc[name]
    tot = row.fn_erosion_km2 + row.fn_standalone_km2
    print(f'{name:<34} erosion {row.fn_erosion_km2:.4f}  standalone {row.fn_standalone_km2:.4f}'
          f'  ({100*row.fn_standalone_km2/tot:.0f}% whole-lot)')

t1.to_csv(RUN / 'table1_overall.csv', index=False)
t2.to_csv(RUN / 'table2_category_removal.csv')
print(f'\nwrote all outputs to {RUN}')

## 12. What is cached, and what a disconnect costs

| Stage | Cached to | Cost if the runtime is recycled |
|---|---|---|
| clone + 100 LFS TIFFs | not cacheable | 10–20 min |
| prepared patches | `RUN/prepared_data/` | ~1 min copy |
| category rasters | `RUN/cache/category_layers.npz` | seconds |
| weight codes | `RUN/weight_codes.zip` | ~30 s unzip |
| training | `RUN/last.ckpt` + `RUN/epochs/` + `targeted_log.csv`, **every epoch** | resumes at the next epoch |
| each evaluation arm | `RUN/eval_cache/*.json` | only the unfinished arm |

So after a drop the only unavoidable cost is the LFS pull. Everything else resumes.

## How to read the result

1. **Arm A must reproduce 0.5190 / 0.8819 / 0.4853.** If not, stop — nothing else is comparable.
2. **Check the gradient mass print in section 8.** Suppression and recovery mass should now be
   close to 1 : 1, against run 1's 4.33 : 1.
3. **Recall, arm C vs arm B.** Run 1 gave 0.6935 against 0.7548. If the rebalance worked, C
   should now be at or above B while keeping the higher precision.
4. **Selectivity gap.** Run 1: B +9.7, C +16.1. The question is whether the gap survives once
   the model is no longer contracting — a gap that only appears under contraction is not
   evidence about categories.
5. **Arm D versus arm C.** How much recall the IoU selection rule costs, with the model fixed.
6. **Whole-lot share of FN.** Run 1's targeted arm put 68% of FN in whole-lot misses. If the
   rebalance works this should fall back toward the generic model's 58%.